# Consultas SQL desde PostgreSQL



In [1]:
import pandas as pd
import psycopg2

# Conexión al PostgreSQL de Docker desde Windows
conn = psycopg2.connect(
    host="127.0.0.1",
    port=5433,
    dbname="metrobus",
    user="metrobus",
    password="metrobus"
)

print("Conexión correcta a PostgreSQL")


Conexión correcta a PostgreSQL


## 1 Cuales son los vehículos mas caros de mantener?


In [3]:
query_1 = """
SELECT
    v.vehiculo_id,
    v.modelo,
    ROUND(SUM(m.coste_eur)::numeric, 2) AS coste_total_mantenimiento
FROM fact_mantenimiento m
JOIN dim_vehiculo v
    ON m.vehiculo_id = v.vehiculo_id
GROUP BY v.vehiculo_id,  v.modelo
ORDER BY coste_total_mantenimiento DESC
LIMIT 15;
"""

df_1 = pd.read_sql(query_1, conn)
df_1

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\1252310641.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_1 = pd.read_sql(query_1, conn)


,vehiculo_id,modelo,coste_total_mantenimiento
0,14,Mercedes-Benz eCitaro,31760.82
1,43,MAN Lions City G,26735.24
2,6,Solaris Urbino 18,25891.04
3,22,BYD eBus-12,24216.84
4,15,Scania Citywide,24184.68
5,8,BYD eBus-12,23684.06
6,33,Iveco Urbanway,23121.76
7,42,MAN Lions City,22832.86
8,17,Mercedes-Benz Citaro,22165.29
9,31,Mercedes-Benz Citaro,20145.96


## 2 Cuales son los vehiculos que mas mantenimientos necesitan?


In [4]:
query_2 = """
SELECT
    v.vehiculo_id,
    v.modelo,
    COUNT(m.mantenimiento_id) AS numero_mantenimientos
FROM fact_mantenimiento m
JOIN dim_vehiculo v
    ON m.vehiculo_id = v.vehiculo_id
GROUP BY v.vehiculo_id,  v.modelo
ORDER BY numero_mantenimientos DESC
LIMIT 15;
"""

df_2 = pd.read_sql(query_2, conn)
df_2

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\562244228.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_2 = pd.read_sql(query_2, conn)


,vehiculo_id,modelo,numero_mantenimientos
0,19,Iveco Urbanway,30
1,18,Iveco Urbanway,30
2,26,Volvo 7900 Hybrid,30
3,22,BYD eBus-12,29
4,6,Solaris Urbino 18,29
5,42,MAN Lions City,29
6,43,MAN Lions City G,28
7,31,Mercedes-Benz Citaro,28
8,14,Mercedes-Benz eCitaro,26
9,13,MAN Lions City G,25


## 3  Cual es la antigüedad media del personal?


In [5]:
query_3 = """
SELECT
    ROUND(AVG(antiguedad_anos)::numeric, 2) AS antiguedad_media_anos
FROM dim_conductor;
"""

df_3 = pd.read_sql(query_3, conn)
df_3

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\3492655853.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_3 = pd.read_sql(query_3, conn)


,antiguedad_media_anos
0,10.5


## 4  Cuantos viajes realiza un conductor al año?


In [6]:
query_4 = """
SELECT
    EXTRACT(YEAR FROM v.fecha)::int AS anio,
    c.conductor_id,
    c.nombre,
    COUNT(DISTINCT v.viaje_id) AS total_viajes
FROM fact_viajes v
JOIN dim_conductor c
    ON v.conductor_id = c.conductor_id
GROUP BY EXTRACT(YEAR FROM v.fecha), c.conductor_id, c.nombre
ORDER BY anio, total_viajes DESC;
"""

df_4 = pd.read_sql(query_4, conn)
df_4

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\3514877239.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_4 = pd.read_sql(query_4, conn)


,anio,conductor_id,nombre,total_viajes
0,2022,10,Carmen Ruiz,637
1,2022,24,Patricia Serrano,633
2,2022,27,Victor Blanco,625
3,2022,23,Raul Ramos,623
4,2022,3,Juan Martinez,622
...,...,...,...,...
79,2024,8,Isabel Perez,577
80,2024,21,Alejandro Castro,576
81,2024,19,Jorge Alonso,568
82,2024,16,Marta Hernandez,564


## 5  Cuantos incidentes tiene un conductor al año?


In [7]:
query_5 = """
SELECT
    EXTRACT(YEAR FROM i.fecha)::int AS anio,
    c.conductor_id,
    c.nombre,
    COUNT(DISTINCT i.incidencia_id) AS total_incidencias
FROM fact_incidencias i
JOIN dim_conductor c
    ON i.conductor_id = c.conductor_id
GROUP BY EXTRACT(YEAR FROM i.fecha), c.conductor_id, c.nombre
ORDER BY anio, total_incidencias DESC;
"""

df_5 = pd.read_sql(query_5, conn)
df_5

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\2932380938.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_5 = pd.read_sql(query_5, conn)


,anio,conductor_id,nombre,total_incidencias
0,2022,24,Patricia Serrano,59
1,2022,22,Beatriz Ortega,56
2,2022,2,Maria Lopez,54
3,2022,9,David Jimenez,52
4,2022,14,Sara Romero,51
...,...,...,...,...
79,2024,6,Laura Gonzalez,43
80,2024,19,Jorge Alonso,42
81,2024,3,Juan Martinez,42
82,2024,28,Silvia Delgado,39


## 6 Cual es el retraso medio de salida por linea?


In [9]:
query_6 = """
SELECT
    l.linea_id,
    l.codigo,
    ROUND(AVG(v.retraso_salida_min)::numeric, 2) AS retraso_medio_min
FROM fact_viajes v
JOIN dim_linea l
    ON v.linea_id = l.linea_id
GROUP BY l.linea_id, l.codigo
ORDER BY retraso_medio_min DESC;
"""

df_6 = pd.read_sql(query_6, conn)
df_6

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\230016477.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_6 = pd.read_sql(query_6, conn)


,linea_id,codigo,retraso_medio_min
0,4,L4,4.44
1,9,L9,4.42
2,3,L3,4.39
3,5,L5,4.34
4,7,L7,4.32
5,8,L8,4.30
6,1,L1,4.29
7,6,L6,4.28
8,2,L2,4.23
9,10,N1,1.53


## 7 Cuales son los tipos de incidentes mas frecuentes?


In [10]:
query_7 = """
SELECT
    tipo_incidencia,
    COUNT(*) AS total_incidencias
FROM fact_incidencias
GROUP BY tipo_incidencia
ORDER BY total_incidencias DESC;
"""

df_7 = pd.read_sql(query_7, conn)
df_7

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\237144204.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_7 = pd.read_sql(query_7, conn)


,tipo_incidencia,total_incidencias
0,Condiciones meteo,382
1,Averia mecanica,347
2,Obras calzada,340
3,Huelga parcial,339
4,Pinchazo,335
5,Accidente leve,332
6,Fallo electrico,332
7,Vandalismo,330
8,Accidente grave,328
9,Fallo climatizacion,312


## 9 Cuales son las paradas de bus con mayor retraso?


In [12]:
query_8 = """
SELECT
    p.parada_id,
    p.nombre_parada,
    COUNT(DISTINCT v.viaje_id) AS total_viajes,
    ROUND(AVG(v.retraso_salida_min)::numeric, 2) AS retraso_medio_min
FROM fact_viajes v
JOIN dim_parada p
    ON v.parada_origen_id = p.parada_id
GROUP BY p.parada_id, p.nombre_parada
ORDER BY retraso_medio_min DESC
LIMIT 15;
"""

df_8 = pd.read_sql(query_8, conn)
df_8

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\1221489465.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_8 = pd.read_sql(query_8, conn)


,parada_id,nombre_parada,total_viajes,retraso_medio_min
0,27,Parada Poligono 27,1225,4.40
1,25,Parada Universidad 25,1184,4.29
2,8,Parada Aeropuerto 8,1228,4.21
3,19,Parada Aeropuerto 19,1204,4.19
4,13,Parada Barrio Sur 13,1240,4.17
5,1,Parada Barrio Norte 1,1251,4.17
6,33,Parada Centro 33,1234,4.17
7,7,Parada Zona Comercial 7,1212,4.14
8,9,Parada Pueblo Oeste 9,1289,4.13
9,32,Parada Pueblo Este 32,1292,4.12


## 9 Cuantos dias pasa al año en mantenimiento un vehiculo en promedio?


In [13]:
query_9 = """
WITH dias_por_vehiculo AS (
    SELECT
        EXTRACT(YEAR FROM fecha_entrada)::int AS anio,
        vehiculo_id,
        SUM(dias_fuera_servicio) AS dias_mantenimiento
    FROM fact_mantenimiento
    GROUP BY EXTRACT(YEAR FROM fecha_entrada), vehiculo_id
)
SELECT
    anio,
    ROUND(AVG(dias_mantenimiento)::numeric, 2)
        AS promedio_dias_mantenimiento_por_vehiculo
FROM dias_por_vehiculo
GROUP BY anio
ORDER BY anio;
"""

df_9 = pd.read_sql(query_9, conn)
df_9

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\4054715572.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_9 = pd.read_sql(query_9, conn)


,anio,promedio_dias_mantenimiento_por_vehiculo
0,2022,12.29
1,2023,13.08
2,2024,15.36
3,2025,34.33


## 10 Cuantos pasajeros lleva cada linea en cada viaje en promedio?


In [15]:
query_10 = """
SELECT
    l.linea_id,
    l.codigo,
    COUNT(DISTINCT v.viaje_id) AS total_viajes,
    ROUND(AVG(v.pasajeros_subidos)::numeric, 2)
        AS pasajeros_promedio_por_viaje
FROM fact_viajes v
JOIN dim_linea l
    ON v.linea_id = l.linea_id
GROUP BY l.linea_id, l.codigo
ORDER BY pasajeros_promedio_por_viaje DESC;
"""

df_10 = pd.read_sql(query_10, conn)
df_10

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\1641260647.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_10 = pd.read_sql(query_10, conn)


,linea_id,codigo,total_viajes,pasajeros_promedio_por_viaje
0,4,L4,4840,64.60
1,9,L9,5024,64.51
2,1,L1,4954,63.98
3,8,L8,5003,63.89
4,2,L2,5073,63.80
5,3,L3,4949,63.67
6,7,L7,5129,63.47
7,5,L5,5128,63.19
8,6,L6,4990,62.81
9,10,N1,4910,38.58


In [16]:
conn.close()
print("Conexión cerrada")


Conexión cerrada
